# Comparación completa de métricas (baseline, v1, v2 y barrido de λ)

Reimplementación independiente de todas las métricas (sin reutilizar el código de los notebooks 13 y 17), concebida como contraste cruzado previo a la interpretación de los resultados de los nuevos valores de λ. El procedimiento de validación es el siguiente:

1. Se ejecuta el notebook incluyendo `baseline`, `v1` y `v2` en la lista de modelos.
2. Se comparan los valores obtenidos con los ya conocidos: Dice 0.7459 / 0.7549 / 0.7482, HD95 crudo 24.02 / 21.06 / 23.35 mm y cobertura 0.4810 / 0.4829 / 0.4798. El HD95 se calcula aquí con `medpy` (percentil 95 del conjunto de distancias entre superficies en ambos sentidos); el notebook 17 utiliza una definición previa (máximo de los dos percentiles direccionales), que arroja valores mayores (35.81 / 33.74 / 33.98 mm) y no es comparable.
3. Si coinciden (salvo redondeo), la lógica de este notebook es consistente con la de los anteriores y los resultados de los nuevos λ son fiables. En caso contrario, la definición de alguna métrica difiere entre notebooks y debe diagnosticarse antes de continuar.

## Métricas calculadas por caso de validación
- **Dice** volumétrico.
- **clDice** (versión dura, con esqueleto real mediante `skimage.morphology.skeletonize`, no la versión diferenciable usada en el entrenamiento).
- **Número de componentes conexas** de la predicción (conectividad 26, adecuada para vasos finos en diagonal).
- **Cobertura de la componente principal**: fracción del vaso de referencia cubierta por la componente conexa más grande de la predicción.
- **HD95** en mm: cruda, restringida a la componente principal y filtrada (se descartan las componentes con menos de `MIN_VOXELS_FILTRO` vóxeles antes del cálculo).

## Comparación estadística
Para cada par (modelo, baseline), test de Wilcoxon y t de Student pareado sobre las mismas 40 validaciones, igual que en el protocolo anterior.


## 1. Entorno

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q SimpleITK medpy scikit-image scipy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 50.8 MB/s eta 0:00:00


In [3]:
import os
from pathlib import Path

BASE = Path('/content/drive/MyDrive/TFM_nnUNet')
NNUNET_PREPROCESSED = BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = BASE / 'nnUNet_results'

DATASET_FOLDER_NAME = 'Dataset501_ImageCAS'
CONFIGURATION = '3d_fullres'
FOLD = 0

GT_FOLDER = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / 'gt_segmentations'

print('Ground truth:', GT_FOLDER)
print('Existe:', GT_FOLDER.exists())

Ground truth: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset501_ImageCAS/gt_segmentations
Existe: True


## 2. Modelos a comparar

La lista de modelos se ajusta según los valores de λ ya entrenados. El `trainer_name` debe coincidir exactamente con el nombre de la carpeta de resultados (`nnUNet_results/Dataset501_ImageCAS/<trainer_name>__nnUNetPlans__3d_fullres/`).


In [4]:
MODELOS = {
    'baseline':  'nnUNetTrainer_100epochs',
    'v1_lam02':  'nnUNetTrainerSoftClDice100epochs',
    'v2_lam02':  'nnUNetTrainerSoftClDiceV2_100epochs',
    'lam01':     'nnUNetTrainerSoftClDice_lam01_100epochs',
    'lam04':     'nnUNetTrainerSoftClDice_lam04_100epochs',
    'lam08':     'nnUNetTrainerSoftClDice_lam08_100epochs',
        # 'lam03':   'nnUNetTrainerSoftClDice_lam03_100epochs',  # se activa al disponer del modelo
        # 'lam05':   'nnUNetTrainerSoftClDice_lam05_100epochs',  # se activa al disponer del modelo
}

def carpeta_validacion(trainer_name):
    return (
        NNUNET_RESULTS
        / DATASET_FOLDER_NAME
        / f'{trainer_name}__nnUNetPlans__{CONFIGURATION}'
        / f'fold_{FOLD}'
        / 'validation'
    )

# Comprobación de que las carpetas existen antes de iniciar el cálculo
for nombre, trainer_name in MODELOS.items():
    carpeta = carpeta_validacion(trainer_name)
    n_casos = len(list(carpeta.glob('*.nii.gz'))) if carpeta.exists() else 0
    estado = 'OK' if n_casos > 0 else 'NO ENCONTRADA / VACÍA'
    print(f'{nombre:12s} -> {carpeta}  [{estado}, {n_casos} archivos .nii.gz]')

baseline     -> /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0/validation  [OK, 40 archivos .nii.gz]
v1_lam02     -> /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainerSoftClDice100epochs__nnUNetPlans__3d_fullres/fold_0/validation  [OK, 40 archivos .nii.gz]
v2_lam02     -> /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainerSoftClDiceV2_100epochs__nnUNetPlans__3d_fullres/fold_0/validation  [OK, 40 archivos .nii.gz]
lam01        -> /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainerSoftClDice_lam01_100epochs__nnUNetPlans__3d_fullres/fold_0/validation  [OK, 40 archivos .nii.gz]
lam04        -> /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainerSoftClDice_lam04_100epochs__nnUNetPlans__3d_fullres/fold_0/validation  [OK, 40 archivos .nii.gz]
lam08        -> /content/drive/MyDrive/TFM_nnUNet/nn

## 3. Funciones de métricas

Si los valores de baseline, v1 y v2 obtenidos en la verificación (sección 6) no coinciden con los resultados previos, estas definiciones son el primer punto a revisar, ya que una diferencia de criterio (p. ej., conectividad 6 frente a 26, o el umbral de filtrado) modifica directamente los números.


In [5]:
import numpy as np
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

MIN_VOXELS_FILTRO = 20  # mismo umbral que en el diagnóstico de HD95 (notebook 17)
CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)  # conectividad completa, adecuada para vasos finos


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]  # SimpleITK da (x,y,z); numpy array está en (z,y,x)
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def filtrar_componentes_pequenas(pred, min_voxels=MIN_VOXELS_FILTRO):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return pred
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    etiquetas_validas = set(np.where(tamanos >= min_voxels)[0] + 1)
    mascara = np.isin(labeled, list(etiquetas_validas))
    return mascara.astype(pred.dtype)


def solo_componente_principal(pred):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return pred
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    return (labeled == idx_mayor).astype(pred.dtype)


def hd95_seguro(pred, gt, spacing):
        # medpy.hd95 genera un error si alguna de las dos máscaras está vacía
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)

    dice = dice_score(pred, gt)
    cldice = hard_cldice(pred, gt)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)

    hd95_crudo = hd95_seguro(pred, gt, spacing)
    hd95_principal = hd95_seguro(solo_componente_principal(pred), gt, spacing)
    hd95_filtrado = hd95_seguro(filtrar_componentes_pequenas(pred), gt, spacing)

    return {
        'dice': dice,
        'cldice': cldice,
        'n_componentes': n_componentes,
        'cobertura_principal': cobertura,
        'hd95_crudo_mm': hd95_crudo,
        'hd95_principal_mm': hd95_principal,
        'hd95_filtrado_mm': hd95_filtrado,
    }

## 4. Verificación con un caso sintético (antes de procesar 40 casos × N modelos)

Se emplea un tubo cilíndrico (sección circular) en lugar de un cubo sólido. El `skeletonize()` 3D (algoritmo de Lee) degenera a un esqueleto vacío sobre sólidos de caras completamente planas, una limitación conocida del algoritmo sobre formas sin curvatura y no un error de la función `hard_cldice`. Dado que un vaso real tiene sección aproximadamente circular, el tubo constituye la prueba representativa. Se comprobó que, sobre un cubo, clDice resulta 0.0 aunque la predicción sea idéntica a la referencia, un falso negativo del caso de prueba y no de la métrica.

Resultados esperados: Dice y clDice iguales a 1.0, HD95 igual a 0.0 y una única componente conexa. Si alguna comprobación falla, debe revisarse la función antes de evaluar los 40 casos reales.


In [6]:
zz, yy, xx = np.mgrid[0:40, 0:15, 0:15]
sintetico = (((yy - 7) ** 2 + (xx - 7) ** 2) <= 4 ** 2).astype(np.uint8)

metrica_test = {
    'dice': dice_score(sintetico, sintetico),
    'cldice': hard_cldice(sintetico, sintetico),
    'cobertura': cobertura_componente_principal(sintetico, sintetico)[0],
    'n_componentes': cobertura_componente_principal(sintetico, sintetico)[1],
    'hd95': hd95_seguro(sintetico, sintetico, (1.0, 1.0, 1.0)),
}

print(metrica_test)

assert abs(metrica_test['dice'] - 1.0) < 1e-6
assert abs(metrica_test['cldice'] - 1.0) < 1e-6
assert abs(metrica_test['cobertura'] - 1.0) < 1e-6
assert metrica_test['n_componentes'] == 1
assert metrica_test['hd95'] == 0.0
print('\nOK: métricas correctas sobre el caso sintético trivial (predicción = referencia).')

# Segundo caso sintético: dos cubos separados, para comprobar la detección de 2 componentes
sintetico2 = np.zeros((30, 30, 30), dtype=np.uint8)
sintetico2[2:5, 2:5, 2:5] = 1
sintetico2[20:25, 20:25, 20:25] = 1
_, n2 = componentes_conexas(sintetico2)
assert n2 == 2
print('OK: detecta correctamente 2 componentes conexas separadas.')

{'dice': np.float64(0.999999999997449), 'cldice': np.float64(0.9999999947222222), 'cobertura': np.float64(0.999999999994898), 'n_componentes': 1, 'hd95': np.float64(0.0)}

OK: métricas correctas sobre el caso sintético trivial (predicción = referencia).
OK: detecta correctamente 2 componentes conexas separadas.


## 5. Cálculo de métricas para todos los modelos y casos

El cálculo requiere varios minutos (HD95 y skeletonize sobre volúmenes 3D grandes). Se guarda un CSV incremental para conservar los resultados ante una posible desconexión de Colab.


In [7]:
import pandas as pd

CSV_RESULTADOS = BASE / 'comparacion_metricas_completo.csv'

if CSV_RESULTADOS.exists():
    df_existente = pd.read_csv(CSV_RESULTADOS)
    print(f'CSV existente encontrado con {len(df_existente)} filas — se completará lo que falte.')
else:
    df_existente = pd.DataFrame(columns=['modelo', 'case_id'])

filas = df_existente.to_dict('records')
ya_calculado = {(f['modelo'], f['case_id']) for f in filas}

for nombre_modelo, trainer_name in MODELOS.items():
    carpeta_pred = carpeta_validacion(trainer_name)
    if not carpeta_pred.exists():
        print(f'[{nombre_modelo}] carpeta no encontrada, se omite: {carpeta_pred}')
        continue

    archivos_pred = sorted(carpeta_pred.glob('*.nii.gz'))
    print(f'[{nombre_modelo}] {len(archivos_pred)} casos encontrados')

    for pred_path in archivos_pred:
        case_id = pred_path.name.replace('.nii.gz', '')
        if (nombre_modelo, case_id) in ya_calculado:
            continue

        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            print(f'  [{nombre_modelo}/{case_id}] sin ground truth correspondiente, se omite')
            continue

        try:
            metricas = calcular_metricas_caso(pred_path, gt_path)
            metricas['modelo'] = nombre_modelo
            metricas['case_id'] = case_id
            filas.append(metricas)
        except Exception as e:
            print(f'  [{nombre_modelo}/{case_id}] ERROR: {e}')

        # Guardado incremental tras cada modelo completo
    pd.DataFrame(filas).to_csv(CSV_RESULTADOS, index=False)
    print(f'[{nombre_modelo}] completo, guardado en {CSV_RESULTADOS}')

df = pd.DataFrame(filas)
print('\nTotal de filas:', len(df))
df.head()

[baseline] 40 casos encontrados
[baseline] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv
[v1_lam02] 40 casos encontrados
[v1_lam02] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv
[v2_lam02] 40 casos encontrados
[v2_lam02] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv
[lam01] 40 casos encontrados
[lam01] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv
[lam04] 40 casos encontrados
[lam04] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv
[lam08] 40 casos encontrados
[lam08] completo, guardado en /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_completo.csv

Total de filas: 240


,dice,cldice,n_componentes,cobertura_principal,hd95_crudo_mm,hd95_principal_mm,hd95_filtrado_mm,modelo,case_id
0,0.697345,0.771060,17,0.451368,48.562174,56.053484,48.569353,baseline,ImageCAS_0011
1,0.711042,0.750254,17,0.339008,31.607630,54.793977,31.514846,baseline,ImageCAS_0017
2,0.709800,0.781705,15,0.525682,36.518146,67.521088,36.519183,baseline,ImageCAS_0018
3,0.609545,0.714437,34,0.517066,62.818912,67.054876,62.820910,baseline,ImageCAS_0021
4,0.785743,0.821244,5,0.390265,4.666234,72.693179,5.076285,baseline,ImageCAS_0028


## 6. Resumen por modelo y contraste con los resultados previos

Con `baseline`, `v1_lam02` y `v2_lam02` en la lista de la sección 2, la tabla reproduce (salvo redondeo) los valores ya conocidos: Dice 0.7459 / 0.7549 / 0.7482 y HD95 crudo medio 24.02 / 21.06 / 23.35 mm (definición de `medpy`).


In [8]:
resumen = df.groupby('modelo').agg(
    dice_mean=('dice', 'mean'),
    cldice_mean=('cldice', 'mean'),
    n_componentes_mean=('n_componentes', 'mean'),
    cobertura_mean=('cobertura_principal', 'mean'),
    hd95_crudo_mean=('hd95_crudo_mm', 'mean'),
    hd95_principal_mean=('hd95_principal_mm', 'mean'),
    hd95_filtrado_mean=('hd95_filtrado_mm', 'mean'),
).round(4)

resumen

,dice_mean,cldice_mean,n_componentes_mean,cobertura_mean,hd95_crudo_mean,hd95_principal_mean,hd95_filtrado_mean
modelo,,,,,,,
baseline,0.7459,0.8069,17.150,0.4810,24.0244,59.6199,24.0524
lam01,0.7428,0.8016,17.450,0.4833,24.4737,60.4414,24.4616
lam04,0.7537,0.8185,15.125,0.4718,20.1636,61.4079,20.1545
lam08,0.7401,0.8085,15.625,0.4838,23.6521,60.9716,23.6462
v1_lam02,0.7549,0.8221,14.725,0.4829,21.0636,61.0859,21.0751
v2_lam02,0.7482,0.8094,16.100,0.4798,23.3501,60.6592,23.3644


## 7. Comparaciones pareadas: cada modelo frente al baseline (Wilcoxon y t-test)


In [9]:
from scipy import stats

METRICAS_A_TESTEAR = ['dice', 'cldice', 'cobertura_principal', 'hd95_crudo_mm']

pivote = {}
for metrica in METRICAS_A_TESTEAR:
    pivote[metrica] = df.pivot(index='case_id', columns='modelo', values=metrica)

resultados_tests = []
for metrica in METRICAS_A_TESTEAR:
    tabla = pivote[metrica]
    if 'baseline' not in tabla.columns:
        print(f'[{metrica}] no hay columna baseline, se omite comparación')
        continue
    for modelo in tabla.columns:
        if modelo == 'baseline':
            continue
        pareja = tabla[['baseline', modelo]].dropna()
        if len(pareja) < 5:
            continue
        try:
            w_stat, w_p = stats.wilcoxon(pareja['baseline'], pareja[modelo])
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        t_stat, t_p = stats.ttest_rel(pareja['baseline'], pareja[modelo])
        resultados_tests.append({
            'metrica': metrica,
            'modelo_vs_baseline': modelo,
            'n': len(pareja),
            'media_baseline': pareja['baseline'].mean(),
            'media_modelo': pareja[modelo].mean(),
            'wilcoxon_p': w_p,
            'ttest_p': t_p,
            'significativo_ambos_p<0.05': (w_p < 0.05) and (t_p < 0.05),
        })

df_tests = pd.DataFrame(resultados_tests).round(4)
df_tests

,metrica,modelo_vs_baseline,n,media_baseline,media_modelo,wilcoxon_p,ttest_p,significativo_ambos_p<0.05
0,dice,lam01,40,0.7459,0.7428,0.0622,0.1615,False
1,dice,lam04,40,0.7459,0.7537,0.0001,0.0001,True
2,dice,lam08,40,0.7459,0.7401,0.0041,0.0139,True
3,dice,v1_lam02,40,0.7459,0.7549,0.0000,0.0000,True
4,dice,v2_lam02,40,0.7459,0.7482,0.3682,0.2682,False
5,cldice,lam01,40,0.8069,0.8016,0.0548,0.0658,False
6,cldice,lam04,40,0.8069,0.8185,0.0000,0.0001,True
7,cldice,lam08,40,0.8069,0.8085,0.6850,0.5832,False
8,cldice,v1_lam02,40,0.8069,0.8221,0.0000,0.0000,True
9,cldice,v2_lam02,40,0.8069,0.8094,0.3203,0.3473,False


## 8. Dice y clDice en función de λ (compromiso entre detección y Dice)


In [10]:
# Asignación manual modelo -> valor de λ (se amplía al añadir modelos)
LAMBDA_POR_MODELO = {
    'baseline': 0.0,
    'lam01': 0.1,
    'v1_lam02': 0.2,
    'lam04': 0.4,
    'v2_lam02': 0.2,   # v2 también usa λ=0.2 pero con distinta implementación; considera excluirlo de este gráfico
    'lam08': 0.8,
}

resumen_lam = resumen.reset_index()
resumen_lam['lambda'] = resumen_lam['modelo'].map(LAMBDA_POR_MODELO)
resumen_lam = resumen_lam.dropna(subset=['lambda']).sort_values('lambda')
resumen_lam[['modelo', 'lambda', 'dice_mean', 'cldice_mean', 'cobertura_mean', 'hd95_crudo_mean']]

,modelo,lambda,dice_mean,cldice_mean,cobertura_mean,hd95_crudo_mean
0,baseline,0.0,0.7459,0.8069,0.4810,24.0244
1,lam01,0.1,0.7428,0.8016,0.4833,24.4737
5,v2_lam02,0.2,0.7482,0.8094,0.4798,23.3501
4,v1_lam02,0.2,0.7549,0.8221,0.4829,21.0636
2,lam04,0.4,0.7537,0.8185,0.4718,20.1636
3,lam08,0.8,0.7401,0.8085,0.4838,23.6521


## 9. Exportación para la memoria del TFM


In [11]:
CSV_RESUMEN_FINAL = BASE / 'resumen_barrido_lambda.csv'
resumen_lam.to_csv(CSV_RESUMEN_FINAL, index=False)
df_tests.to_csv(BASE / 'tests_pareados_barrido_lambda.csv', index=False)

print('Guardado:')
print(' -', CSV_RESUMEN_FINAL)
print(' -', BASE / 'tests_pareados_barrido_lambda.csv')

Guardado:
 - /content/drive/MyDrive/TFM_nnUNet/resumen_barrido_lambda.csv
 - /content/drive/MyDrive/TFM_nnUNet/tests_pareados_barrido_lambda.csv
